# **SET UP** เริ่มต้นด้วยการดาวน์โหลด ก่อนเสมอ

In [ ]:
pip install google-generativeai

#Mini hackathon สัปดาห์ที่ 2: รอบแรก CELL CODE
***ปล.เนื่องจากผมทำใน kaggle ไม่ใช่ colab อาจจะต้องมีการปรับเปลี่ยนโค้ดอะไรนิดหน่อย***

In [ ]:
import os
import re
import json
import difflib
import time
from collections import defaultdict
from PIL import Image
import pandas as pd
from google import genai
from google.genai import types
from kaggle_secrets import UserSecretsClient

# ==========================================
# 1. ตั้งค่า API และโมเดล
# ==========================================
try:
    user_secrets = UserSecretsClient()
    my_api_key = user_secrets.get_secret("GEMINI_API_KEY")
    client = genai.Client(api_key=my_api_key)
except:
    print("❌ อย่าลืมตั้งค่า GEMINI_API_KEY")

MODEL_ID = 'gemini-3.1-flash-lite-preview'

# ==========================================
# 2. 🌟 ไม้ตายที่ 2: Chain-of-Thought Prompts
# ==========================================
# บังคับให้ AI พ่น JSON แบบมี "raw_text" เพื่อยืนยันสิ่งที่เห็นก่อนตอบ
CONSTITUENCY_PROMPT = """
You are an expert OCR and Data Extraction AI processing ONE PAGE of a Thai Constituency Election Document.
Extract the exact vote count for each political party from the provided image.

CRITICAL RULES:
1. Extract ONLY "Party Name" and "Votes". Ignore Candidate Names.
2. HANDWRITTEN CORRECTIONS: If a printed number is crossed out, extract the handwritten value.
3. FORMAT: Convert Thai numerals to Arabic. Remove commas.

OUTPUT FORMAT: Return ONLY a valid JSON object. For each party, output a nested object with 'raw_text' (exactly what you see) and 'final_score'.
Example:
{
  "พรรคก้าวไกล": {
    "raw_text": "๑๒,๓๔๕ (หนึ่งหมื่นสองพันสามร้อยสี่สิบห้า)",
    "final_score": "12345"
  }
}
"""

PARTY_LIST_PROMPT = """
You are an expert OCR and Data Extraction AI processing ONE PAGE of a Thai Party List Election Document.
Extract the exact vote count for each political party from the provided image.

CRITICAL RULES:
1. Extract ONLY "Party Name" and "Votes".
2. HANDWRITTEN CORRECTIONS: If printed number is crossed out, extract the handwritten value.
3. FORMAT: Convert Thai numerals to Arabic. Remove commas.

OUTPUT FORMAT: Return ONLY a valid JSON object. For each party, output a nested object with 'raw_text' and 'final_score'.
Example:
{
  "พรรคเพื่อไทย": {
    "raw_text": "๕๐๐ (ห้าร้อย)",
    "final_score": "500"
  }
}
"""

# ==========================================
# 3. จัดกลุ่มไฟล์
# ==========================================
IMAGE_DIR = "/kaggle/input/competitions/super-ai-engineer-season-6-ocr-2569/data/images/"
all_files = os.listdir(IMAGE_DIR) if os.path.exists(IMAGE_DIR) else []
grouped_docs = { "constituency": defaultdict(list), "party_list": defaultdict(list) }
file_pattern = re.compile(r"^(constituency|party_list)_(\d+)_(\d+)(?:_page\d+)?\.(png|jpg|jpeg)$", re.IGNORECASE)

for filename in all_files:
    match = file_pattern.match(filename)
    if match:
        doc_type = match.group(1).lower()
        doc_key = f"{match.group(2)}_{match.group(3)}"
        grouped_docs[doc_type][doc_key].append(filename)

for dt in grouped_docs:
    for dk in grouped_docs[dt]:
        grouped_docs[dt][dk].sort()

# ==========================================
# 4. 🌟 ไม้ตายที่ 1: ฟังก์ชันอ่านทีละหน้า (Page-by-Page)
# ==========================================
def extract_single_page(prompt, image_filename):
    img = Image.open(os.path.join(IMAGE_DIR, image_filename))
    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=[prompt, img],
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                temperature=0.0
            )
        )
        return json.loads(response.text)
    except Exception as e:
        print(f"  [Error in page {image_filename}]: {e}")
        return {}

# ==========================================
# 5. ลูปรันงาน (พร้อมระบบ Merge ข้อมูล)
# ==========================================
final_all_results = { "constituency": {}, "party_list": {} }

print(f"🚀 เริ่มดึงข้อมูลระดับ Ultimate (ร่อนขยะแบบละเอียด)...")

for doc_type in ["constituency", "party_list"]:
    prompt = CONSTITUENCY_PROMPT if doc_type == "constituency" else PARTY_LIST_PROMPT

    for doc_key, pages in grouped_docs[doc_type].items():
        print(f"กำลังสแกน {doc_type} เขต {doc_key} ({len(pages)} หน้า)...")

        merged_doc_result = {}

        for page_img in pages:
            page_json = extract_single_page(prompt, page_img)

          # ... โค้ดด้านบนของข้อ 5 เหมือนเดิม ...
            for party, data in page_json.items():

                # 1. ทำความสะอาดชื่อพรรคพื้นฐาน
                clean_p = str(party).replace("พรรค", "").replace(" ", "").strip()

                # 🛠️ 2. ดักจับขยะ Header/Footer (ใช้ได้กับทั้ง 2 แบบ)
                noise_words = ["รวมคะแนน", "รวมทั้งสิ้น", "บัตรเสีย", "ไม่ประสงค์", "ลายมือ", "ผู้สมัคร", "หมายเลข", "ได้คะแนน"]
                if any(noise in clean_p for noise in noise_words) or clean_p == "รวม" or clean_p == "รวมคะแนน":
                    continue

                # 🛠️ 3. ดักจับชื่อคน (*** บังคับใช้เฉพาะ Constituency เท่านั้น ***)
                if doc_type == "constituency":
                    # เติมจุดให้ยศ ป้องกันการไปเตะพรรค "พลัง..." ทิ้ง
                    prefixes = ["นาย", "นาง", "น.ส.", "นางสาว", "ร.ต.", "จ.ส.", "พล.", "พ.ต.", "ส.ต.", "ด.ต."]
                    if any(clean_p.startswith(p) for p in prefixes):
                        continue

                # 4. ดึงคะแนนออกมา
                if isinstance(data, dict):
                    score = data.get("final_score", "0")
                else:
                    score = str(data)

                # 5. เก็บลงตะกร้า
                if clean_p not in merged_doc_result and clean_p != "":
                    merged_doc_result[clean_p] = score

            time.sleep(4)
            # ... โค้ดด้านล่างเหมือนเดิม ...

print("✅ ดึงข้อมูล AI แบบทีละหน้าเสร็จสมบูรณ์!")
# ==========================================
# 6. Submission + 🌟 ไม้ตายที่ 3: กันเหนียวเลขไทย
# ==========================================
print("📝 กำลังจับคู่คำและสร้างไฟล์ Submission...")
TEMPLATE_PATH = "/kaggle/input/competitions/super-ai-engineer-season-6-ocr-2569/data/submission_template.csv"
submission_df = pd.read_csv(TEMPLATE_PATH)

def clean_party_name(name):
    return str(name).replace("พรรค", "").replace(" ", "").strip()

# ตารางแปลงเลขไทย -> อารบิก แบบ Hardcode ใน Python (ปลอดภัย 100%)
thai_to_arabic = str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789')

def get_predicted_votes_ultimate(row):
    try:
        doc_id_str = str(row['doc_id']).strip()
        target_name_clean = clean_party_name(row['party_name'])

        doc_type = "party_list" if doc_id_str.startswith("party") else "constituency"
        parts = doc_id_str.split('_')
        doc_key = f"{parts[2]}_{parts[3]}" if doc_type == "party_list" else f"{parts[1]}_{parts[2]}"

        doc_data = final_all_results.get(doc_type, {}).get(doc_key, {})
        if not doc_data: return "0"

        extracted_keys_clean = {clean_party_name(k): k for k in doc_data.keys()}
        raw_val = "0"

        # Exact + Fuzzy Match
        if target_name_clean in extracted_keys_clean:
            raw_val = doc_data[extracted_keys_clean[target_name_clean]]
        else:
            matches = difflib.get_close_matches(target_name_clean, extracted_keys_clean.keys(), n=1, cutoff=0.60) # ลด cutoff ลงนิดนึงให้จับได้กว้างขึ้น
            if matches:
                raw_val = doc_data[extracted_keys_clean[matches[0]]]

        # ทำความสะอาดขั้นสุดยอด (ลบลูกน้ำ, แปลงเลขไทยที่หลงมา, เช็คตัวเลข)
        clean_value = str(raw_val).translate(thai_to_arabic).replace(",", "").replace(" ", "").strip()

        # กรองเอาเฉพาะตัวเลขจริงๆ ป้องกันตัวอักษรหลุดรอด
        clean_value = ''.join(filter(str.isdigit, clean_value))

        return clean_value if clean_value else "0"

    except Exception as e:
        return "0"

submission_df['votes'] = submission_df.apply(get_predicted_votes_ultimate, axis=1)

# กรอง 2 คอลัมน์และเซฟไฟล์
final_submission_df = submission_df[['id', 'votes']]
output_filename = "document_level_submission_V3_ULTIMATE.csv"
final_submission_df.to_csv(output_filename, index=False)

print(f"🎉 สร้างไฟล์ {output_filename} สำเร็จ! ไปทุบกำแพง 0.0925 กันเลยครับ!!")


#Mini hackathon สัปดาห์ที่ 2: รอบสอง CELL CODE
***ปล.เนื่องจากผมทำใน kaggle ไม่ใช่ colab อาจจะต้องมีการปรับเปลี่ยนโค้ดอะไรนิดหน่อย***

In [ ]:
import os
import re
import json
import time
import difflib
from collections import defaultdict
from PIL import Image, ImageEnhance
import pandas as pd
from google import genai
from google.genai import types
from kaggle_secrets import UserSecretsClient
from pydantic import BaseModel, Field

# ==========================================
# 1. ตั้งค่า API และ โมเดล (Global Pool 8 API Keys แบบจัดเต็ม!)
# ==========================================
print("🚀 เริ่มต้นรันโค้ดแบบ All-in-One (Ultimate Pydantic + Fuzzy Match Version)...")
try:
    user_secrets = UserSecretsClient()

    api_key1 = user_secrets.get_secret("GEMINI_API_KEY")
    api_key2 = user_secrets.get_secret("GEMINI_API_KEY2")
    api_key3 = user_secrets.get_secret("GEMINI_API_KEY3")
    api_key4 = user_secrets.get_secret("GEMINI_API_KEY4")
    api_key5 = user_secrets.get_secret("GEMINI_API_KEY5")
    api_key6 = user_secrets.get_secret("GEMINI_API_KEY6")
    api_key7 = user_secrets.get_secret("GEMINI_API_KEY7")
    api_key8 = user_secrets.get_secret("GEMINI_API_KEY8")

    client1 = genai.Client(api_key=api_key1)
    client2 = genai.Client(api_key=api_key2)
    client3 = genai.Client(api_key=api_key3)
    client4 = genai.Client(api_key=api_key4)
    client5 = genai.Client(api_key=api_key5)
    client6 = genai.Client(api_key=api_key6)
    client7 = genai.Client(api_key=api_key7)
    client8 = genai.Client(api_key=api_key8)

    print("✅ โหลด API Key ทั้ง 8 ตัวสำเร็จ ขุมพลังพร้อมลุย!")
except Exception as e:
    print(f"❌ พบปัญหาการโหลด API Key โปรดตรวจสอบ Kaggle Secrets: {e}")

MODEL_ID = 'gemini-3.1-flash-lite-preview'
GLOBAL_POOL = [client1, client2, client3, client4, client5, client6, client7, client8]

import os
import re
import json
import time
from collections import defaultdict
from PIL import Image, ImageEnhance
import pandas as pd
from google import genai
from google.genai import types
from kaggle_secrets import UserSecretsClient

# ==========================================
# 1. ตั้งค่า API และ โมเดล (Global Pool 4 API Keys)
# ==========================================
print("🚀 เริ่มต้นรันโค้ดแบบ All-in-One (Rollback to Stable XML JSON Version)...")
try:
    user_secrets = UserSecretsClient()

    api_key1 = user_secrets.get_secret("GEMINI_API_KEY")
    api_key2 = user_secrets.get_secret("GEMINI_API_KEY2")
    api_key3 = user_secrets.get_secret("GEMINI_API_KEY3")
    api_key4 = user_secrets.get_secret("GEMINI_API_KEY4")

    client1 = genai.Client(api_key=api_key1)
    client2 = genai.Client(api_key=api_key2)
    client3 = genai.Client(api_key=api_key3)
    client4 = genai.Client(api_key=api_key4)

    print("✅ โหลด API Key ทั้ง 4 ตัวสำเร็จ ขุมพลังพร้อมลุย!")
except Exception as e:
    print(f"❌ พบปัญหาการโหลด API Key โปรดตรวจสอบ Kaggle Secrets: {e}")

MODEL_ID = 'gemini-3.1-flash-lite-preview'
GLOBAL_POOL = [client1, client2, client3, client4]

# ==========================================
# 2. ฟังก์ชันปรับรูปภาพ (Image Preprocessing)
# ==========================================
def preprocess_image_for_ai(image_path):
    img = Image.open(image_path)
    enhancer_contrast = ImageEnhance.Contrast(img)
    img_contrast = enhancer_contrast.enhance(1.5)
    enhancer_sharpness = ImageEnhance.Sharpness(img_contrast)
    return enhancer_sharpness.enhance(2.0)

# ==========================================
# 3. XML Prompt (นำ <output_format> กลับมาใช้เพื่อบังคับ JSON)
# ==========================================
CONSTITUENCY_PROMPT = """<prompt>
    <role>You are a highly advanced OCR and Document Layout Analysis AI specializing in complex Thai election tally sheets.</role>
    <task>Extract the "Candidate Number" (หมายเลขประจำตัวผู้สมัคร) and their corresponding "Votes" (ได้คะแนน) from the provided image.</task>
    <context>The image contains a handwritten or printed tally table. The document structure contains ชื่อผู้สมัคร + พรรค + คะแนน. The layout might be messy, skewed, or have misaligned rows.</context>
    <rules>
        <rule_1_table_mapping>CRITICAL: Do not just read text left-to-right. You MUST accurately pair the Candidate Number (no) with its exact corresponding Votes (score) based on the table's row structure.</rule_1_table_mapping>
        <rule_2_number_formats>Numbers may NOT be in standard formats. They could be messy handwriting, Thai numerals (๐-๙). Observe carefully, interpret the true value, and convert EVERYTHING to standard Arabic numerals (0-9).</rule_2_number_formats>
        <rule_3_cleaning>Remove all commas, spaces, and stray marks.</rule_3_cleaning>
        <rule_4_blank_cells>If a vote cell is explicitly blank, crossed out, or empty, output "0" for the score.</rule_4_blank_cells>
        <rule_5_exclusions>STRICTLY IGNORE candidate names and party names. Only extract the Number and Votes. IGNORE the summary row usually labeled "รวมคะแนนทั้งสิ้น".</rule_5_exclusions>
    </rules>
    <output_format>-
        Output STRICTLY as a JSON array of objects. Do not wrap in markdown tags. Just the raw array.
        [
            {"no": "1", "score": "1234"},
            {"no": "2", "score": "0"}
        ]
    </output_format>
</prompt>"""

PARTY_LIST_PROMPT = """<prompt>
    <role>You are a highly advanced OCR and Document Layout Analysis AI specializing in complex Thai Party List election tally sheets.</role>
    <task>Extract the "Party Number" (หมายเลข) and their corresponding "Votes" (ได้คะแนน) from the provided image.</task>
    <context>The document contains ชื่อพรรค + คะแนน. The layout is often long and rows might be visually ambiguous.</context>
    <rules>
        <rule_1_table_mapping>CRITICAL: Accurately map the Party Number (no) to its exact Votes (score).</rule_1_table_mapping>
        <rule_2_number_formats>Numbers may appear in non-standard formats. Carefully decode the visual information and convert all valid numeric inputs to standard Arabic numerals (0-9).</rule_2_number_formats>
        <rule_3_cleaning>Remove all commas, spaces, and formatting artifacts.</rule_3_cleaning>
        <rule_4_blank_cells>If a vote cell is blank, contains only a dash, or is crossed out, output "0".</rule_4_blank_cells>
        <rule_5_exclusions>STRICTLY IGNORE party names entirely. SKIP summary rows such as "รวมคะแนนทั้งสิ้น" or "บัตรเสีย".</rule_5_exclusions>
    </rules>
    <output_format>
        Output STRICTLY as a JSON array of objects. Do not wrap in markdown tags. Just the raw array.
        [
            {"no": "1", "score": "10"},
            {"no": "2", "score": "20"}
        ]
    </output_format>
</prompt>"""

# ==========================================
# 4. จัดกลุ่มไฟล์
# ==========================================
IMAGE_DIR = "/kaggle/input/competitions/super-ai-engineer-season-6-ocr-2569-round2/final_data/images"
all_files = os.listdir(IMAGE_DIR) if os.path.exists(IMAGE_DIR) else []
grouped_docs = { "constituency": defaultdict(list), "party_list": defaultdict(list) }
file_pattern = re.compile(r"^(constituency|party_list)_(\d+)_(\d+)(?:_page(\d+))?\.(png|jpg|jpeg)$", re.IGNORECASE)

for filename in all_files:
    match = file_pattern.match(filename)
    if match:
        doc_type = match.group(1).lower()
        doc_key = f"{match.group(2)}_{match.group(3)}"
        grouped_docs[doc_type][doc_key].append(filename)

def get_page_num(filename):
    match = re.search(r'_page(\d+)', filename, re.IGNORECASE)
    return int(match.group(1)) if match else 1

for dt in grouped_docs:
    for dk in grouped_docs[dt]:
        grouped_docs[dt][dk].sort(key=get_page_num)

# ==========================================
# 5. ฟังก์ชันสกัดข้อมูล (ลบ Pydantic ออก ใช้ Regex ดัก JSON แทน)
# ==========================================
current_pool_idx = 0

def extract_single_page_with_retry(prompt, image_filename, max_retries=10):
    global current_pool_idx
    image_path = os.path.join(IMAGE_DIR, image_filename)
    img_for_ai = preprocess_image_for_ai(image_path)

    for attempt in range(max_retries):
        if len(GLOBAL_POOL) == 0:
            print(f"    💀 โควต้าหมดเกลี้ยงทุก KEY แล้วครับ! ขอข้ามไฟล์ {image_filename}")
            return []

        active_client = GLOBAL_POOL[current_pool_idx]

        try:
            # 🚀 นำ response_schema ออก แล้วใช้แค่ mime_type เพื่อความปลอดภัยสูงสุด
            response = active_client.models.generate_content(
                model=MODEL_ID,
                contents=[prompt, img_for_ai],
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=0.0
                )
            )

            raw_text = response.text.strip()
            # ทำความสะอาด Markdown tags เผื่อ AI หลอนส่งมา
            raw_text = re.sub(r'^```(json)?\n?|```$', '', raw_text, flags=re.IGNORECASE).strip()
            data = json.loads(raw_text)

            current_pool_idx = (current_pool_idx + 1) % len(GLOBAL_POOL)
            return data if isinstance(data, list) else []

        except Exception as e:
            error_msg = str(e).lower()
            if attempt < max_retries - 1:
                if "quota" in error_msg:
                    print(f"    🚨 [Quota Empty] โควต้ารายวันหมดเกลี้ยง! ถอดออกจากระบบ...")
                    GLOBAL_POOL.pop(current_pool_idx)
                    if len(GLOBAL_POOL) > 0:
                        current_pool_idx = current_pool_idx % len(GLOBAL_POOL)
                        time.sleep(1)
                elif "429" in error_msg:
                    current_pool_idx = (current_pool_idx + 1) % len(GLOBAL_POOL)
                    print(f"    🔄 [Too Many Requests] สลับเวรไปคีย์ถัดไป...")
                    time.sleep(4)
                elif "503" in error_msg:
                    print("    ⏳ [503] เซิร์ฟเวอร์รันไม่ทัน ขอพัก 30 วินาที...")
                    time.sleep(30)
                else:
                    print(f"    ⚠️ [Retry {attempt+1}] Error: {error_msg[:60]}")
                    time.sleep(5)
            else:
                print(f"    ❌ ข้ามไฟล์ {image_filename} เนื่องจาก Error ซ้ำๆ")
                return []

# ==========================================
# 6. ลูปรันงานกวาดตัวเลข
# ==========================================
final_all_results = { "constituency": {}, "party_list": {} }
print(f"🚀 เริ่มดึงข้อมูลระดับ Ultimate (Stable V4 Template Mapping)...")

thai_to_arabic = str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789')

for doc_type in ["party_list", "constituency"]:
    prompt = CONSTITUENCY_PROMPT if doc_type == "constituency" else PARTY_LIST_PROMPT

    for doc_key, pages in grouped_docs[doc_type].items():
        print(f"กำลังสแกน {doc_type} เขต {doc_key} ({len(pages)} หน้า)...")
        merged_doc_result = {}

        for page_img in pages:
            page_data = extract_single_page_with_retry(prompt, page_img)

            for item in page_data:
                if not isinstance(item, dict): continue

                # 🌟 ดึงหมายเลข ตาม Output Format ของเรา
                raw_no = str(item.get("no", ""))
                data_score = str(item.get("score", "0"))

                clean_no = raw_no.translate(thai_to_arabic).strip()
                clean_no = ''.join(filter(str.isdigit, clean_no))

                clean_score = data_score.translate(thai_to_arabic).replace(",", "").replace(" ", "").strip()
                clean_score = ''.join(filter(str.isdigit, clean_score))
                if not clean_score: clean_score = "0"

                if clean_no and clean_no not in merged_doc_result:
                    merged_doc_result[clean_no] = clean_score

            time.sleep(4)

        final_all_results[doc_type][doc_key] = merged_doc_result
        print(f"   ✔️ เก็บข้อมูล {doc_type} เขต {doc_key} สำเร็จ (ได้ทั้งหมด {len(merged_doc_result)} หมายเลข)")

# ==========================================
# 7. สร้างไฟล์ Submission V4
# ==========================================
print("📝 กำลังจับคู่หมายเลขและสร้างไฟล์ Submission...")

TEMPLATE_PATH = "/kaggle/input/competitions/super-ai-engineer-season-6-ocr-2569-round2/final_data/submission_template_v4.csv"
submission_df = pd.read_csv(TEMPLATE_PATH)

def get_predicted_votes_v4(row):
    try:
        row_id = str(row['id']).strip()
        parts = row_id.split('_')

        candidate_no = parts[-1]

        if row_id.startswith("party_list"):
            doc_type = "party_list"
            doc_key = f"{parts[2]}_{parts[3]}"
        else:
            doc_type = "constituency"
            doc_key = f"{parts[1]}_{parts[2]}"

        doc_data = final_all_results.get(doc_type, {}).get(doc_key, {})

        raw_val = doc_data.get(candidate_no, "0")
        return raw_val

    except Exception as e:
        return "0"

submission_df['votes'] = submission_df.apply(get_predicted_votes_v4, axis=1)

final_submission_df = submission_df[['id', 'votes']]
output_filename = "document_level_submission_V4_FINAL_STABLE.csv"
final_submission_df.to_csv(output_filename, index=False)

print(f"🎉 เสร็จสิ้นทุกกระบวนการ! ไฟล์ {output_filename} ถูกสร้างเรียบร้อย นำไป Submit ได้เลยครับ!")